# ProstT5 Speculative Decoding - Inverse Folding (3Di -> AA)
enc-dec timings loaded from baseline notebook | checkpointing after each protein | greedy only

In [ ]:
%pip install -q "transformers==4.46.3" "accelerate>=0.26.0" sentencepiece pandas matplotlib tqdm

In [ ]:
%matplotlib inline
import os,time,statistics,gc,pickle,json
from pathlib import Path; from datetime import datetime
import numpy as np,torch,torch.nn as nn
from tqdm.auto import tqdm; import pandas as pd,matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":120,"font.size":10})
from transformers import T5Tokenizer,AutoModelForSeq2SeqLM,GenerationConfig,PreTrainedModel
from transformers.modeling_outputs import ModelOutput
try:
    from google.colab import drive; drive.mount("/content/drive")
    USE_DRIVE=True; DRIVE_ROOT="/content/drive/MyDrive"
except Exception:
    USE_DRIVE=False; DRIVE_ROOT=None
device=torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch {torch.__version__} | Device: {device}")

In [ ]:
for _v in ["model","encoder","cnn","cnn_assistant"]:
    if _v in globals(): del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 1. Configuration

In [ ]:
PROSTT5_NAME="Rostlab/ProstT5_fp16"; NOTEBOOK_DIR=Path(".").resolve()
AA_LETTERS="ACDEFGHIKLMNPQRSTVWY"; NUM_REPEATS=5
import matplotlib.cm as _cm
PALETTE={k:_cm.get_cmap("tab10")(i%10) for i,k in enumerate(range(1,17))}
BLUE,ORANGE,GRAY="#2e86de","#f39c12","#7f8c8d"
RESULTS_DIR=NOTEBOOK_DIR.joinpath("benchmark_results"); RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_PKL=RESULTS_DIR.joinpath("invfolding_spec_dec_checkpoint.pkl")
RESULTS_CSV=RESULTS_DIR.joinpath("invfolding_spec_dec_results.csv")
PREDICTIONS_JSON=RESULTS_DIR.joinpath("invfolding_spec_dec_predictions.json")
ALPHA_CSV=RESULTS_DIR.joinpath("invfolding_alpha_results.csv")
def _readable(p):
    try: open(p,"rb").read(4); return True
    except OSError: return False
CNN_CKPT=None
_cands=[NOTEBOOK_DIR.joinpath("model.pt"),NOTEBOOK_DIR.joinpath("cnn_chkpnt_AA_CNN","model.pt")]
if USE_DRIVE: _cands=[Path(DRIVE_ROOT).joinpath("models","CNN_from3di_toAA.pt"),Path(DRIVE_ROOT).joinpath("model.pt")]+_cands
for c in _cands:
    if _readable(c): CNN_CKPT=c; break
if CNN_CKPT is None: raise FileNotFoundError("CNN checkpoint not found.")
print(f"Results: {RESULTS_DIR} | CNN: {CNN_CKPT}")

# Drive subfolder for invfolding spec-dec results
DRIVE_RESULTS = (Path(DRIVE_ROOT) / "models" / "invfolding_spec_dec_results") if USE_DRIVE else None
if DRIVE_RESULTS: DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
if "tokenizer" not in globals():
    tokenizer=T5Tokenizer.from_pretrained(PROSTT5_NAME,do_lower_case=False,legacy=True)
if "model" not in globals():
    dtype=torch.float16 if device.type=="cuda" else torch.float32
    model=AutoModelForSeq2SeqLM.from_pretrained(PROSTT5_NAME,low_cpu_mem_usage=True,torch_dtype=dtype).to(device).eval()
encoder=model.get_encoder()
class InvFoldingCNN(nn.Module):
    def __init__(self,nt=20,h=32,d=1024):
        super().__init__(); self.classifier=nn.Sequential(nn.Conv2d(d,h,(7,1),padding=(3,0)),nn.ReLU(),nn.Dropout(0.0),nn.Conv2d(h,nt,(7,1),padding=(3,0)))
    def forward(self,x): x=x.permute(0,2,1).unsqueeze(-1); x=self.classifier(x); return x.squeeze(-1).permute(0,2,1)
if "cnn" not in globals():
    cnn=InvFoldingCNN().to(device).eval(); ck=torch.load(CNN_CKPT,map_location=device,weights_only=False); cnn.load_state_dict(ck.get("state_dict",ck),strict=True)
AA_TOKEN_IDS=[tokenizer.encode(f" {aa}",add_special_tokens=False)[0] for aa in AA_LETTERS]
DECODER_START=model.config.decoder_start_token_id
print(f"CNN: {sum(p.numel() for p in cnn.parameters()):,} params | Decoder start: {DECODER_START}")

## 2. Dataset

In [ ]:
def parse_fasta(path):
    seqs,cur={},None
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith(">"): cur=line[1:].split()[0]; seqs[cur]=""
        elif cur: seqs[cur]+=line.strip()
    return seqs
DATA_DIR=None
for cand in [NOTEBOOK_DIR.joinpath("benchmark_data"),Path("/content/benchmark_data")]:
    if cand.joinpath("test_set_3Di.fasta").exists(): DATA_DIR=cand; break
if USE_DRIVE and DATA_DIR is None:
    for cand in [Path(DRIVE_ROOT).joinpath("models"),Path(DRIVE_ROOT).joinpath("benchmark_data")]:
        if cand.joinpath("test_set_3Di.fasta").exists(): DATA_DIR=cand; break
if DATA_DIR is None: raise FileNotFoundError("test_set_3Di.fasta not found. Run baseline notebook first.")
di_seqs=parse_fasta(DATA_DIR.joinpath("test_set_3Di.fasta"))
aa_seqs=parse_fasta(DATA_DIR.joinpath("test_set_AA.fasta"))
ALL_PROTEINS=[{"id":uid,"3Di":di_seqs[uid].lower(),"AA":aa_seqs[uid]} for uid in di_seqs if uid in aa_seqs]
print(f"Loaded {len(ALL_PROTEINS)} proteins from {DATA_DIR}")

## 3. Load Baseline enc-dec Timings

In [ ]:
_bc=[RESULTS_DIR.joinpath("summary_per_protein.csv"),Path("/content/benchmark_results/summary_per_protein.csv")]
if USE_DRIVE: _bc+=[
    Path(DRIVE_ROOT).joinpath("models","benchmark_results","summary_per_protein.csv"),
    Path(DRIVE_ROOT).joinpath("models","summary_per_protein.csv"),
    Path(DRIVE_ROOT).joinpath("prostT5_benchmarks","summary_per_protein.csv"),
    Path(DRIVE_ROOT).joinpath("summary_per_protein.csv")]
baseline_summary,encdec_time=None,{}
for p in _bc:
    if p.exists(): baseline_summary=pd.read_csv(p); print(f"Baseline loaded: {p}"); break
if baseline_summary is None:
    print("WARNING: summary_per_protein.csv not found - run prostT5_baseline_performance.ipynb first.")
else:
    _ed=baseline_summary[baseline_summary["pipeline"]=="enc_dec"]
    encdec_time=dict(zip(_ed["protein_id"],_ed["latency_s_median"]))
    print(f"enc-dec timings for {len(encdec_time)} proteins.")

## 4. Hyperparameters

In [ ]:
# ── Edit these before running ──────────────────────────────────────
K_VALUES     = [1, 2, 4, 8]   # draft lengths to sweep
NUM_PROTEINS = 20              # number of proteins to benchmark (max 100, set to None for all)

benchmark_data = ALL_PROTEINS[:NUM_PROTEINS] if NUM_PROTEINS else ALL_PROTEINS
print(f"K_VALUES={K_VALUES} | proteins={len(benchmark_data)} | repeats={NUM_REPEATS}")

## 5. CNNAssistantModel

In [ ]:
class CNNAssistantModel(PreTrainedModel):
    """Inverse-folding CNN (3Di->AA) wrapped as HF-compatible assistant for speculative decoding."""
    config_class=type(model.config)
    def __init__(self,config,enc,cnn_h,ids):
        super().__init__(config); self._encoder=enc; self._cnn=cnn_h; self._aa_token_ids=ids
        self.config.is_encoder_decoder=True; self.config.decoder_start_token_id=config.decoder_start_token_id
        self.generation_config=GenerationConfig(num_assistant_tokens=5,num_assistant_tokens_schedule="constant",do_sample=False,max_length=3000)
        self._cached_logits=None; self._cached_hidden_id=None
    def _full_vocab_logits(self,eo):
        h=eo[0][:,1:-1,:]; l20=self._cnn(h.float())[0]
        full=torch.full((l20.shape[0],self.config.vocab_size),-1e4,device=l20.device,dtype=l20.dtype)
        for i,tid in enumerate(self._aa_token_ids): full[:,tid]=l20[:,i]
        return full
    def get_encoder(self): return self._encoder
    def prepare_inputs_for_generation(self,di,encoder_outputs=None,attention_mask=None,**kw):
        return {"decoder_input_ids":di,"encoder_outputs":encoder_outputs,"attention_mask":attention_mask}
    def forward(self,decoder_input_ids=None,encoder_outputs=None,attention_mask=None,**kw):
        if encoder_outputs is not None:
            hid=id(encoder_outputs[0])
            if self._cached_hidden_id!=hid: self._cached_logits=self._full_vocab_logits(encoder_outputs); self._cached_hidden_id=hid
        n=decoder_input_ids.shape[1]
        if self._cached_logits is not None:
            nr=min(n,self._cached_logits.shape[0]); lg=self._cached_logits[:nr].unsqueeze(0)
            if nr<n: lg=torch.cat([lg,torch.full((1,n-nr,lg.shape[-1]),-1e4,device=lg.device,dtype=lg.dtype)],dim=1)
        else: lg=torch.zeros(1,n,self.config.vocab_size,device=device,dtype=torch.float32)
        return ModelOutput(logits=lg)
if "cnn_assistant" not in globals():
    cnn_assistant=CNNAssistantModel(config=model.config,enc=encoder,cnn_h=cnn,ids=AA_TOKEN_IDS).to(device).eval()
print(f"CNNAssistantModel ready | is_enc_dec={cnn_assistant.config.is_encoder_decoder}")

## 6. Helpers & Checkpointing

In [ ]:
# ── Timing helpers ───────────────────────────────────────────────
def _sync():
    if device.type=="cuda": torch.cuda.synchronize()
    elif device.type=="mps": torch.mps.synchronize()
def _format_3di(seq): return "<fold2AA> "+" ".join(list(seq.lower()))
def _decode_aa(ids):
    ids=ids if isinstance(ids,list) else ids.tolist()
    return "".join(tokenizer.decode(ids,skip_special_tokens=True).split()).upper()

@torch.inference_mode()
def run_encdec(di_seq):
    """Plain enc-dec greedy. Used ONLY in sanity check — NOT called in main benchmark loop."""
    L=len(di_seq); enc=tokenizer([_format_3di(di_seq)],add_special_tokens=True,return_tensors="pt").to(device)
    kw=dict(input_ids=enc.input_ids,attention_mask=enc.attention_mask,max_length=L+2,do_sample=False,num_beams=1)
    times=[]
    for _ in range(NUM_REPEATS):
        _sync(); t0=time.perf_counter(); out=model.generate(**kw); _sync(); times.append(time.perf_counter()-t0)
    return _decode_aa(out[0])[:L], statistics.median(times), float(np.std(times))

@torch.inference_mode()
def run_hf_assisted(di_seq,K):
    """Greedy speculative decoding via HF assistant_model API."""
    L=len(di_seq); enc=tokenizer([_format_3di(di_seq)],add_special_tokens=True,return_tensors="pt").to(device)
    cnn_assistant.generation_config.num_assistant_tokens=K
    kw=dict(input_ids=enc.input_ids,attention_mask=enc.attention_mask,max_length=L+2,do_sample=False,num_beams=1,assistant_model=cnn_assistant)
    times=[]
    for _ in range(NUM_REPEATS):
        _sync(); t0=time.perf_counter(); out=model.generate(**kw); _sync(); times.append(time.perf_counter()-t0)
    return _decode_aa(out[0])[:L], statistics.median(times), float(np.std(times))

# ── Checkpointing ─────────────────────────────────────────────────────
import shutil as _shutil_ckpt

def save_checkpoint(state):
    state["timestamp"]=datetime.now().isoformat()
    with open(CHECKPOINT_PKL,"wb") as f: pickle.dump(state,f)
    if USE_DRIVE:
        _shutil_ckpt.copy(CHECKPOINT_PKL, DRIVE_RESULTS / CHECKPOINT_PKL.name)

def load_checkpoint():
    if CHECKPOINT_PKL.exists():
        with open(CHECKPOINT_PKL,"rb") as f: return pickle.load(f)
    # Fallback: restore from Drive on fresh Colab session
    if USE_DRIVE:
        _drive_pkl = DRIVE_RESULTS / CHECKPOINT_PKL.name
        if _drive_pkl.exists():
            _shutil_ckpt.copy(_drive_pkl, CHECKPOINT_PKL)
            print(f"Restored checkpoint from Drive: {_drive_pkl.name}")
            with open(CHECKPOINT_PKL,"rb") as f: return pickle.load(f)
    return None

# ── Resume from checkpoint ────────────────────────────────────────────
_ckpt=load_checkpoint()
if _ckpt is not None:
    results=_ckpt.get("results",[])
    predictions=_ckpt.get("predictions",{})
    completed=set(_ckpt.get("completed",[]))
    alpha_results=_ckpt.get("alpha_results",{})
    print(f"Resumed: {len(completed)} proteins done, {len(results)} result rows.")
else:
    results,predictions,completed,alpha_results=[],{},set(),{}
    print("Starting fresh — no checkpoint found.")
print("Helpers and checkpointing ready.")

## 7. Sanity Check
Greedy speculative decoding must produce identical output to plain enc-dec greedy. Verified on 5 proteins.

In [ ]:
print("Sanity check on 5 shortest proteins...")
_test=sorted(benchmark_data,key=lambda r: len(r["3Di"]))[:5]
_all_ok=True
for item in _test:
    uid,di_seq=item["id"],item["3Di"]; ref,*_=run_encdec(di_seq)
    for K in K_VALUES:
        try:
            pred,*_=run_hf_assisted(di_seq,K); ok=pred==ref
            if not ok: _all_ok=False
            print(f"  {"PASS" if ok else "FAIL"}  {uid} L={len(di_seq)}  K={K}")
        except Exception as e: print(f"  ERROR  {uid} K={K}: {e}"); _all_ok=False
print("ALL CORRECT" if _all_ok else "FAILURES DETECTED")

## 8. Benchmark — Greedy Speculative Decoding (HF Only)
enc-dec is NOT re-run. Speedup = t_encdec_from_baseline / t_hf.
Results are checkpointed after each protein — restart the kernel and re-run to resume.

In [ ]:
print(f"Benchmarking {len(benchmark_data)} proteins x {len(K_VALUES)} K values (HF assisted only)")
print(f"Proteins already done: {len(completed)}/{len(benchmark_data)}")
_remaining=[item for item in benchmark_data if item["id"] not in completed]
print(f"Remaining: {len(_remaining)}\n")

for item in tqdm(_remaining,desc="Proteins"):
    uid,di_seq=item["id"],item["3Di"]; L=len(di_seq)
    t_baseline=encdec_time.get(uid,None)
    if t_baseline is None:
        print(f"  WARNING: no baseline enc-dec time for {uid} (speedup will be NaN)")
    preds_k={}
    for K in K_VALUES:
        try:
            pred,t_hf,t_hf_std=run_hf_assisted(di_seq,K); hf_ok=True
        except Exception as e:
            print(f"  {uid} K={K}: {e}"); pred,t_hf,t_hf_std,hf_ok="",float("nan"),float("nan"),False
        results.append({"id":uid,"length":L,"K":K,
            "t_encdec_baseline":t_baseline,"t_hf":t_hf,"t_hf_std":t_hf_std if hf_ok else float("nan"),
            "speedup":t_baseline/t_hf if t_baseline and hf_ok and t_hf>0 else float("nan"),
            "hf_ok":hf_ok})
        if hf_ok: preds_k[str(K)]=pred
    predictions[uid]={"3Di":di_seq,"AA":item["AA"],"preds_by_K":preds_k}
    completed.add(uid)
    save_checkpoint({"results":results,"predictions":predictions,"completed":list(completed),"alpha_results":alpha_results})
    gc.collect()
    if device.type=="cuda": torch.cuda.empty_cache()

df=pd.DataFrame(results)
df.to_csv(RESULTS_CSV,index=False)
with open(PREDICTIONS_JSON,"w") as f: json.dump(predictions,f)
print(f"\nDone. {len(df)} rows | {RESULTS_CSV.name} + {PREDICTIONS_JSON.name}")
import shutil as _sh
if USE_DRIVE:
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    _sh.copy(RESULTS_CSV,      DRIVE_RESULTS / RESULTS_CSV.name)
    _sh.copy(PREDICTIONS_JSON, DRIVE_RESULTS / PREDICTIONS_JSON.name)
    print(f"Saved to Drive: invfolding_spec_dec_results/")

## 9. Alpha Analysis (Acceptance Rate)
Checkpointed per protein — re-run to continue from where it stopped.

In [ ]:
_alpha_todo=[item for item in benchmark_data if item["id"] not in alpha_results]
print(f"Alpha: {len(alpha_results)} done, {len(_alpha_todo)} remaining.")

@torch.inference_mode()
def compute_alpha(di_seq):
    L=len(di_seq); enc=tokenizer([_format_3di(di_seq)],add_special_tokens=True,return_tensors="pt").to(device)
    out=model.generate(input_ids=enc.input_ids,attention_mask=enc.attention_mask,max_length=L+2,do_sample=False,num_beams=1)
    fwd=model(input_ids=enc.input_ids,attention_mask=enc.attention_mask,decoder_input_ids=out[:,:-1])
    p_logits=fwd.logits[0,:L]
    enc_h=encoder(input_ids=enc.input_ids,attention_mask=enc.attention_mask).last_hidden_state
    q_logits=cnn(enc_h[:,1:-1,:].float())[0]; L_eff=min(L,p_logits.shape[0],q_logits.shape[0])
    p_p=torch.softmax(p_logits[:L_eff].float(),dim=-1); q_p20=torch.softmax(q_logits[:L_eff].float(),dim=-1)
    q_p=torch.zeros_like(p_p)
    for i,tid in enumerate(AA_TOKEN_IDS): q_p[:,tid]=q_p20[:,i]
    a=torch.minimum(p_p,q_p).sum(dim=-1); return float(a.mean()),float(a.std())

for item in tqdm(_alpha_todo,desc="Alpha"):
    uid,di_seq=item["id"],item["3Di"]
    try: am,as_=compute_alpha(di_seq)
    except Exception as e: print(f"  {uid}: {e}"); am,as_=float("nan"),float("nan")
    alpha_results[uid]={"alpha_mean":am,"alpha_std":as_,"length":len(di_seq)}
    save_checkpoint({"results":results,"predictions":predictions,"completed":list(completed),"alpha_results":alpha_results})

alpha_df=pd.DataFrame([{"id":k,**v} for k,v in alpha_results.items()])
alpha_df.to_csv(ALPHA_CSV,index=False)
print(f"Alpha saved: {ALPHA_CSV.name}")
import shutil as _sh
if USE_DRIVE:
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    _sh.copy(ALPHA_CSV, DRIVE_RESULTS / ALPHA_CSV.name)
    print(f"Saved to Drive: invfolding_spec_dec_results/")
_am=alpha_df.alpha_mean; print(f"alpha mean={_am.mean():.4f} std={_am.std():.4f} min={_am.min():.4f} max={_am.max():.4f}")

## 10. Load Results for Analysis
Run this cell after the benchmark and alpha cells. Works without loading the model — reads from saved CSV/JSON.

In [ ]:
# Load from CSV/JSON — re-run this block any time without the model
if RESULTS_CSV.exists():
    df=pd.read_csv(RESULTS_CSV); print(f"Results: {len(df)} rows from {RESULTS_CSV.name}")
elif results:
    df=pd.DataFrame(results); print(f"Results: {len(df)} rows (in-memory)")
else:
    raise RuntimeError("No results found. Run the benchmark first.")

if ALPHA_CSV.exists():
    alpha_df=pd.read_csv(ALPHA_CSV); print(f"Alpha: {len(alpha_df)} proteins from {ALPHA_CSV.name}")
elif alpha_results:
    alpha_df=pd.DataFrame([{"id":k,**v} for k,v in alpha_results.items()]); print("Alpha: in-memory")
else:
    print("WARNING: No alpha results. Run alpha computation first.")
    alpha_df=pd.DataFrame(columns=["id","alpha_mean","alpha_std","length"])

preds_loaded={}
if PREDICTIONS_JSON.exists():
    with open(PREDICTIONS_JSON,"r") as f: preds_loaded=json.load(f)
    print(f"Predictions: {len(preds_loaded)} proteins from {PREDICTIONS_JSON.name}")
elif predictions:
    preds_loaded=predictions; print(f"Predictions: {len(preds_loaded)} proteins (in-memory)")

df=df.merge(alpha_df[["id","alpha_mean","alpha_std"]],on="id",how="left")
def _pred_tps(a,K): return (1-a**(K+1))/(1-a) if (not pd.isna(a)) and a<0.9999 else K+1
df["pred_tps"]=df.apply(lambda r: _pred_tps(r["alpha_mean"],r["K"]),axis=1)
print(f"\nReady: {df['id'].nunique()} proteins, {len(df)} rows.")

In [ ]:
print("="*65)
print("BENCHMARK RESULTS -- ProstT5 Speculative Decoding (3Di -> AA)")
print("="*65)
_nu=df["id"].nunique(); print(f"Proteins: {_nu} | Device: {device}")
print("K  | alpha  | Pred tok/step | Speedup mean | HF OK%")
print("-"*60)
for K in K_VALUES:
    sub=df[df["K"]==K]; a=sub.alpha_mean.mean(); pt=sub.pred_tps.mean()
    sp=sub.speedup.mean(); ok=sub.hf_ok.mean()*100
    print(f"{K:2d} | {a:.4f} | {pt:13.3f} | {sp:12.3f}x | {ok:6.1f}%")
print("="*65)
print()
print("Predicted vs Measured (Theorem 3.8):")
print("K  | Pred tok/step | Speedup mean | Ratio")
print("-"*45)
for K in K_VALUES:
    sub=df[df["K"]==K]; pred=sub.pred_tps.mean(); meas=sub.speedup.mean()
    r=meas/pred if pred>0 else float("nan")
    print(f"{K:2d} | {pred:13.3f} | {meas:12.3f}x | {r:.3f}")
print("Ratio < 1: CNN prefix-independence reduces effective acceptance")

In [ ]:
def seq_rec(pred,gt):
    n=min(len(pred),len(gt)); return sum(p==g for p,g in zip(pred,gt))/n if n else 0.0
_bk=K_VALUES[int(df.groupby("K").speedup.mean().values.argmax())]
rec_rows=[]
for item in benchmark_data:
    uid,gt=item["id"],item["AA"]
    if uid not in preds_loaded: continue
    _p=preds_loaded[uid].get("preds_by_K",{})
    rec_rows.append({"id":uid,"length":len(gt),
        "rec_best_K":seq_rec(_p.get(str(_bk),""),gt)})
rec_df=pd.DataFrame(rec_rows)
print(f"Sequence recovery vs ground-truth AA (best K={_bk}):")
_rb=rec_df.rec_best_K
print(f"  HF K={_bk}: {_rb.mean():.3f} +/- {_rb.std():.3f}")
print("  (All K values produce identical output for correct greedy speculative decoding)")

## 11. Plots

In [ ]:
import matplotlib.cm as _cm2
fig,axes=plt.subplots(2,3,figsize=(18,10))
axes=axes.flatten()
valid_K=[K for K in K_VALUES if df[df["K"]==K]["speedup"].notna().any()]
ax=axes[0]
bp=ax.boxplot([df[df["K"]==K]["speedup"].dropna().values for K in valid_K],patch_artist=True,widths=0.5,medianprops={"color":"black","lw":2},showmeans=True,meanprops={"marker":"D","markerfacecolor":"white","markeredgecolor":"black","markersize":6})
for patch,K in zip(bp["boxes"],valid_K): patch.set_facecolor(PALETTE.get(K,BLUE)); patch.set_alpha(0.75)
ax.axhline(1.0,color="red",linestyle="--",alpha=0.5,label="no speedup"); ax.set_xticklabels([f"K={K}" for K in valid_K])
ax.set_ylabel("Speedup over enc-dec"); ax.set_title("Speedup by K"); ax.legend(fontsize=8); ax.grid(True,alpha=0.3,axis="y")
ax=axes[1]; _av=alpha_df.alpha_mean.dropna()
ax.hist(_av,bins=20,color=BLUE,alpha=0.8,edgecolor="white"); _ma,_sa=_av.mean(),_av.std()
ax.axvline(_ma,color="red",linestyle="--",lw=2,label=f"mean={_ma:.3f}")
ax.axvspan(_ma-_sa,_ma+_sa,alpha=0.12,color="red"); ax.set_title("Alpha distribution"); ax.set_xlim(0,1); ax.legend(fontsize=8); ax.grid(True,alpha=0.3,axis="y")
ax=axes[2]; _ma2=alpha_df.alpha_mean.mean(); K_arr=np.array(K_VALUES)
pt_arr=[(1-_ma2**(K+1))/(1-_ma2) if _ma2<0.9999 else K+1 for K in K_arr]
ms=[df[df["K"]==K]["speedup"].mean() for K in K_arr]; msd=[df[df["K"]==K]["speedup"].std() for K in K_arr]
x,w=np.arange(len(K_arr)),0.35
ax.bar(x-w/2,pt_arr,w,color=BLUE,alpha=0.85,label="Pred (Thm 3.8)")
ax.bar(x+w/2,ms,w,color=ORANGE,alpha=0.85,label="Measured",yerr=msd,capsize=5,error_kw={"elinewidth":1.5,"ecolor":"black"})
ax.set_xticks(x); ax.set_xticklabels([f"K={K}" for K in K_arr]); ax.set_title("Pred vs Measured"); ax.legend(fontsize=8); ax.grid(True,alpha=0.3,axis="y")
ax=axes[3]; sub0=df[df["K"]==K_VALUES[0]]
ax.scatter(sub0["length"],sub0["t_encdec_baseline"],color=GRAY,alpha=0.6,s=25,label="enc-dec baseline",zorder=3)
for K in K_VALUES:
    sub=df[df["K"]==K].dropna(subset=["t_hf"])
    _yerr=sub["t_hf_std"].fillna(0) if "t_hf_std" in sub.columns else None
    ax.errorbar(sub["length"],sub["t_hf"],yerr=_yerr,fmt="o",markersize=4,alpha=0.5,capsize=3,color=PALETTE.get(K,BLUE),label=f"HF K={K}")
ax.set_yscale("log"); ax.set_xscale("log"); ax.set_title("Wall Time vs Length"); ax.legend(fontsize=7,ncol=2); ax.grid(True,alpha=0.3,which="both")
ax=axes[4]
bk=K_VALUES[int(np.nanargmax([df[df["K"]==K]["speedup"].mean() for K in K_VALUES]))]
sub=df[df["K"]==bk].dropna(subset=["speedup","alpha_mean"])
sc=ax.scatter(sub["length"],sub["speedup"],c=sub["alpha_mean"],cmap="RdYlGn",s=40,alpha=0.85,edgecolors="none")
ax.axhline(1.0,color="red",linestyle="--",alpha=0.4); fig.colorbar(sc,ax=ax,label="alpha")
ax.set_title(f"Speedup vs Length (K={bk})"); ax.grid(True,alpha=0.3)
ax=axes[5]
for K in K_VALUES:
    sub=df[df["K"]==K].dropna(subset=["speedup"]).sort_values("length")
    if "t_hf_std" in sub.columns:
        _yerr=sub["speedup"]/sub["t_hf"].replace(0,float("nan"))*sub["t_hf_std"].fillna(0)
    else:
        _yerr=None
    ax.errorbar(sub["length"],sub["speedup"],yerr=_yerr,fmt="o",alpha=0.5,markersize=4,capsize=2,color=PALETTE.get(K,BLUE),label=f"K={K}")
ax.axhline(1.0,color="red",linestyle="--",alpha=0.4); ax.set_title("Per-Protein Speedup"); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
fig.suptitle("ProstT5 Speculative Decoding -- Inverse Folding (3Di -> AA)",fontsize=13,y=1.01)
plt.tight_layout()
_pl=RESULTS_DIR.joinpath("invfolding_spec_dec_plots.png"); plt.savefig(_pl,bbox_inches="tight",dpi=150)
import shutil as _sh
if USE_DRIVE:
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    _sh.copy(_pl, DRIVE_RESULTS / "invfolding_spec_dec_plots.png")
    print(f"Saved to Drive: invfolding_spec_dec_results/invfolding_spec_dec_plots.png")
plt.show(); print(f"Saved: {_pl.name}")

In [ ]:
mean_alpha=alpha_df.alpha_mean.mean()
print("="*65)
print("SUMMARY -- ProstT5 Speculative Decoding, Inverse Folding (3Di -> AA)")
print("="*65)
_sp_by_k=[df[df["K"]==K]["speedup"].mean() for K in K_VALUES]
best_K=K_VALUES[int(np.nanargmax(_sp_by_k))]; best_sp=df[df["K"]==best_K]["speedup"].mean()
pred_best=(1-mean_alpha**(best_K+1))/(1-mean_alpha) if mean_alpha<0.9999 else best_K+1
print(f"  Proteins         : {df['id'].nunique()}")
print(f"  Device           : {device}")
print(f"  Drafter CNN      : {sum(p.numel() for p in cnn.parameters()):,} params")
print(f"  Mean alpha       : {mean_alpha:.4f}")
print()
print(f"{'K':>3}  {'alpha':>7}  {'Pred tok/step':>14}  {'Speedup mean':>13}  {'HF OK%':>7}")
print("-"*55)
for K in K_VALUES:
    sub=df[df["K"]==K]; a=sub.alpha_mean.mean(); pt=sub.pred_tps.mean()
    sp=sub.speedup.mean(); ok=sub.hf_ok.mean()*100
    print(f"{K:3d}  {a:7.4f}  {pt:14.3f}  {sp:13.3f}x  {ok:6.1f}%")
print("="*65)
print()
print(f"  Best K: {best_K}  ->  mean speedup {best_sp:.2f}x")
print(f"  Theorem 3.8 predicts {pred_best:.2f} tok/step for K={best_K}; measured speedup is {best_sp:.2f}x.")
print(f"  Speedup bound: 1/(1-alpha) = {1/(1-mean_alpha):.2f}x regardless of K.")